# 01 Data Preprocessing

This notebook prepares the Airline Passenger Satisfaction dataset for supervised machine learning.

The workflow includes:

- Loading train and test data
- Removing non-informative columns
- Handling missing values
- Encoding categorical variables
- Scaling numerical features
- Creating final train/test matrices

## Imports

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
except ImportError:
    print("Please install xgboost: pip install xgboost")

RANDOM_STATE = 42

## Load Data

In [ ]:
# Update these paths if your CSV files are stored elsewhere.
train_path = "../data/train.csv"
test_path = "../data/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Remove unnamed index columns if present
train_df = train_df.loc[:, ~train_df.columns.str.contains("^Unnamed")]
test_df = test_df.loc[:, ~test_df.columns.str.contains("^Unnamed")]

display(train_df.head())
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

## Missing Value Treatment

In [ ]:
print("Missing values before imputation:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

arrival_delay_median = train_df["Arrival Delay in Minutes"].median()
train_df["Arrival Delay in Minutes"] = train_df["Arrival Delay in Minutes"].fillna(arrival_delay_median)
test_df["Arrival Delay in Minutes"] = test_df["Arrival Delay in Minutes"].fillna(arrival_delay_median)

print("\nMissing values after imputation:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

## Categorical Encoding

In [ ]:
nominal_cols = ["Gender", "Type of Travel"]
ordinal_cols = ["Customer Type", "Class"]

ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
oe = OrdinalEncoder(
    categories=[
        ["disloyal Customer", "Loyal Customer"],
        ["Eco", "Eco Plus", "Business"]
    ]
)
le = LabelEncoder()

train_nominal = pd.DataFrame(
    ohe.fit_transform(train_df[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=train_df.index
)

test_nominal = pd.DataFrame(
    ohe.transform(test_df[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=test_df.index
)

train_ordinal = pd.DataFrame(
    oe.fit_transform(train_df[ordinal_cols]),
    columns=ordinal_cols,
    index=train_df.index
)

test_ordinal = pd.DataFrame(
    oe.transform(test_df[ordinal_cols]),
    columns=ordinal_cols,
    index=test_df.index
)

train_encoded = train_df.drop(columns=nominal_cols + ordinal_cols).join(train_nominal).join(train_ordinal)
test_encoded = test_df.drop(columns=nominal_cols + ordinal_cols).join(test_nominal).join(test_ordinal)

train_encoded["satisfaction"] = le.fit_transform(train_encoded["satisfaction"])
test_encoded["satisfaction"] = le.transform(test_encoded["satisfaction"])

display(train_encoded.head())

## Feature Matrix and Scaling

In [ ]:
X_train = train_encoded.drop(columns=["satisfaction", "id"])
y_train = train_encoded["satisfaction"]

X_test = test_encoded.drop(columns=["satisfaction", "id"])
y_test = test_encoded["satisfaction"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Final feature columns:")
print(list(X_train.columns))